In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
os.environ["DIRECTORY_PATH"] = os.getenv("DIRECTORY_PATH")

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")
output = model.invoke("hi")
output.content

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text',
  'text': 'Hello! How can I help you today?',
  'extras': {'signature': 'EvAFCu0FAWkUfRMR/Bqgqw+DnKK29XCiNPjo0fekc/jix+dmQ18Z/w8M5DjU1EwxuXab+JQ9X0Hepjxq4Lolw6fbJ/r28vT1pyAHiyeen8b6W310G0p+wVuVKR6bbkBs1jjFFnWaZ1Jaw6ZpiVhBhsJtVuKm0+fM3dUAkHRnPte9Mvvqz/RM7IJrHV1brChdYxYv0ldmcIJYc+JUKW6Wue098ayzIM7rdYdQZrL9PEwqnSMum3dYbso0Q17oaW0VVyUhTFVdLsHhZpNJwp/nI13RPsXoQ3SBHBcoFmskCpMyeM99HIylVi/LYvOJu8K5lVBwDSgzM7VyqpYFp9waqYPn9RTysMqgd5g7Ci166W1ni5/LmJ+nR2/T2V4qcufSmGetKTj3MWKZjxzXP34YIdjkpfMzU/nr4476PbToEffJ2Q30k9OIIRtXEIAApOSOC3Qm2gH95WrMWhI0fBXHJj6DXRYosEE1CSTN5QfhZsckCTh48M5uub9Dp1wG/Xd77W+lMIADkl1sBKXG2mwJqg1qtLx2/UvvQIyjeDGckqnUEzDZCMXP/lgDDfUKKwSiS5VXH4UqLEXssJtQVeDzOz18lVEM/pGFAURIFo/Z2/uP1SF0VPfvvpt9gT6EvQ2dBok6BwBHroTz+1Ox8q9NoeAWdm+/qCYjyeWBeFCND9wwXgK4pfDz5nb+I2oV1J5bVLIgC05EqdF6/tv4cnUcGqmqc6l1X9OyJqwg+495umsyaUJ8vb2tJVJcVa8BomPmSqTcWpZssYvr/xwxuBPFTuf3Hs4kpb3Z4WDhg+psqjdhlBarFFMtvgrNZzu+IhHbCIaTL7jcHmR2rm28BoDCDptBFAgGFwsqDYCGvioenizG9Wx5FfaPW3wkdc5OTx945kVXKrzQTvwVP

In [5]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")
len(embeddings.embed_query("Hi"))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


3072

In [6]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Mohit\AppData\Local\Temp\ipykernel_5512\4068926105.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader
d:\03_Study\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
loader = DirectoryLoader(os.environ["DIRECTORY_PATH"] + "\\Source Files", glob= "./*.txt", loader_cls= TextLoader)

In [41]:
docs = loader.load()

In [42]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)

In [43]:
new_docs = text_splitter.split_documents(docs)

In [44]:
doc_chunks = [doc.page_content for doc in new_docs]

In [45]:
len(doc_chunks)

55

In [46]:
db = Chroma.from_documents(new_docs, embeddings)

In [47]:
retriever = db.as_retriever(search_kwargs = {"k" : 2})

In [48]:
retriever.invoke("Industrial Growth of US?")

[Document(metadata={'source': 'D:\\03_Study\\GenAI\\3.3 Traditional RAG 02\\Source Files\\usa.txt'}, page_content='The U.S. maintains its GDP growth through strong innovation, entrepreneurship, and investment in R&D. With companies like Apple, Google, Amazon, Microsoft, and Tesla leading global markets, the U.S.'),
 Document(metadata={'source': 'D:\\03_Study\\GenAI\\3.3 Traditional RAG 02\\Source Files\\usa.txt'}, page_content='The U.S. maintains its GDP growth through strong innovation, entrepreneurship, and investment in R&D. With companies like Apple, Google, Amazon, Microsoft, and Tesla leading global markets, the U.S.')]

In [49]:
import operator
from typing import List
from pydantic import BaseModel , Field
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph,END

In [50]:
from pydantic import BaseModel, Field
class TopicSelectionParser(BaseModel):
    Topic:str = Field("Selected Topic")
    Reasoning:str = Field("Reason of selecting Topic")


In [51]:
from langchain_core.output_parsers import PydanticOutputParser
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator


In [52]:
parser = PydanticOutputParser(pydantic_object = TopicSelectionParser)

In [53]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"default": "Selected Topic", "title": "Topic", "type": "string"}, "Reasoning": {"default": "Reason of selecting Topic", "title": "Reasoning", "type": "string"}}}\n```'

In [54]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage],operator.add]

In [55]:
def function_1(state: AgentState):
    question = state['messages'][-1]
    print(question)
    template="""
    Your task is to classify the given user query into one of the following categories: [USA,Not Related]. 
    Only respond with the category name and nothing else.

    User query: {question}
    {format_instructions}
    """
    prompt= PromptTemplate(
        template=template,
        input_variable=["question"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )

    chain = prompt | model | parser 
    response = chain.invoke({"question" : question})
    return {"messages" : [response.Topic]}

In [56]:
state = {"messages": ["what is today's weather"]}

In [57]:
function_1(state)

what is today's weather


{'messages': ['Not Related']}

In [58]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [59]:
def function_2(state: AgentState):
    question = state["messages"][0]
    prompt=PromptTemplate(
        template="""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:""",
        
        input_variables=['context', 'question']
    )
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | model
        | StrOutputParser()
    )
    result = rag_chain.invoke(question)
    return  {"messages": [result]}

In [60]:
def function_3(state: AgentState):
    question = state["messages"][0]
    
    # Normal LLM call
    complete_query = "Anwer the follow question with you knowledge of the real world. Following is the user question: " + question
    response = model.invoke(complete_query)
    return {"messages": [response.content]}

In [61]:
def router(state: AgentState):
    last_message = state["messages"][-1]
    print(last_message)
    if "usa" in last_message.lower():
        return "RAG Call"
    else:
        return "LLM Call"

In [62]:
from langgraph.graph import StateGraph, START, END

In [63]:
workflow = StateGraph(AgentState)

In [64]:
workflow.add_node("Supervisor",function_1)
workflow.add_node("RAG",function_2)
workflow.add_node("LLM", function_3)


In [65]:
workflow.set_entry_point("Supervisor")
workflow.add_conditional_edges(
            "Supervisor", router , 
            {"RAG Call" : "RAG", "LLM Call" : "LLM"} 
            )
workflow.add_edge("RAG", END)
workflow.add_edge("LLM", END)

In [66]:
app = workflow.compile()

In [ ]:
app.invoke({"messages": ["What is the GDP of USA?"] })